In [1]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

# How to Compute the Conjunctive Normal Form

Formulas are represented as values of the type `Formula`.  In order to convert a string into a formula we use the *parser* that is found in the notebook `Propositional-Logic-Parser.ipynb`.

In [2]:
importNotebook "Propositional-Logic-Parser.ipynb"

imported 

Propositional-Logic-Parser.ipynb

A formula in *conjunctive normal form* is represented in *set notation*, i.e. as a set of *clauses*, where every clause is a set of *literals*.  A literal is either a propositional variable or the negation of a propositional variable.  We define the following types:
- A `Literal` is either `Pos p`, which represents the propositional variable $p$, or `Neg p`, which represents the negated variable $\neg p$.
- A `Clause` is a set of literals.  It represents the disjunction of its literals.
- A `CNF` is a set of clauses.  It represents the conjunction of its clauses.

The keyword `type` followed by a name and a type defines a *type abbreviation*.  For example, `Clause` is just another name for the type `Set<Literal>`.

These types are also used in the notebooks implementing the algorithm of Davis and Putnam.

In [3]:
type Literal =
    | Pos of string   // p
    | Neg of string   // ¬p

type Clause = Set<Literal>
type CNF    = Set<Clause>

The function `eliminateBiconditional` takes one argument:
- `f` is a propositional formula.

It returns a formula that is equivalent to `f` and does not contain the operators `↔` and `⊕`.  This is done by using the following equivalences:
$$ (g \leftrightarrow h) \;\Leftrightarrow\; (g \rightarrow h) \wedge (h \rightarrow g) $$
$$ (g \oplus h) \;\Leftrightarrow\; \neg (g \leftrightarrow h) $$
The Python version of this function does not handle the operator `⊕`.  In F#, the compiler warns us if a pattern match does not cover all constructors of the type `Formula`.  Therefore, we have to deal with `⊕`.

In [4]:
let rec eliminateBiconditional (f: Formula) : Formula =
    match f with
    | Var _ | Verum | Falsum -> f
    | Iff (g, h) -> eliminateBiconditional (And (Imp (g, h), Imp (h, g)))
    | Xor (g, h) -> eliminateBiconditional (Not (Iff (g, h)))
    | Not g      -> Not (eliminateBiconditional g)
    | And (g, h) -> And (eliminateBiconditional g, eliminateBiconditional h)
    | Or  (g, h) -> Or  (eliminateBiconditional g, eliminateBiconditional h)
    | Imp (g, h) -> Imp (eliminateBiconditional g, eliminateBiconditional h)

The function `eliminateConditional` takes one argument:
- `f` is a propositional formula that does not contain the operators `↔` and `⊕`.

It returns a formula that is equivalent to `f` and does not contain the operator `→`.  This is done by using the following equivalence:
$$ (g \rightarrow h) \;\Leftrightarrow\; (\neg g \vee h) $$
If `f` contains `↔` or `⊕`, an exception is raised.

In [5]:
let rec eliminateConditional (f: Formula) : Formula =
    match f with
    | Var _ | Verum | Falsum -> f
    | Imp (g, h) -> eliminateConditional (Or (Not g, h))
    | Not g      -> Not (eliminateConditional g)
    | And (g, h) -> And (eliminateConditional g, eliminateConditional h)
    | Or  (g, h) -> Or  (eliminateConditional g, eliminateConditional h)
    | Iff _ | Xor _ -> failwith "eliminateConditional: ↔ and ⊕ have to be eliminated first"

The function `nnf` takes one argument:
- `f` is a propositional formula that contains only the operators `¬`, `∧`, `∨`, `⊤`, and `⊥`.

It returns the *negation normal form* of `f`.  The auxiliary function `neg` takes the same kind of argument `f` and returns the *negation normal form* of $\neg f$.

The function $\texttt{nnf}$ is defined recursively as follows:
<ol>
    <li> $\texttt{nnf}(\neg \texttt{F}) = \texttt{neg}(\texttt{F})$, </li>
    <li> $\texttt{nnf}(\texttt{F}_1 \wedge \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \wedge \texttt{nnf}(\texttt{F}_2)$,</li>
    <li> $\texttt{nnf}(\texttt{F}_1 \vee \texttt{F}_2) = 
          \texttt{nnf}(\texttt{F}_1) \vee \texttt{nnf}(\texttt{F}_2)$.</li>
</ol>
The auxiliary function $\texttt{neg}$ is also defined recursively:
<ol>
    <li> $\texttt{neg}(p) = \texttt{nnf}(\neg p) = \neg p$ for all propositional variables $p$,</li>
    <li> $\texttt{neg}(\top) = \bot$ and $\texttt{neg}(\bot) = \top$,</li>
    <li> $\texttt{neg}(\neg F) = \texttt{nnf}(\neg \neg F) = \texttt{nnf}(F)$,</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \wedge F_2 \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \wedge F_2)\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \vee \neg F_2\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \vee \texttt{nnf}\bigl(\neg F_2\bigr) \\[0.1cm]
       = & \texttt{neg}(F_1) \vee \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \wedge F_2 \bigr) = \texttt{neg}(F_1) \vee \texttt{neg}(F_2)$.</li>
     <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \vee F_2 \bigr)        \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \vee F_2) \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \wedge \neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \wedge \texttt{nnf}\bigl(\neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{neg}(F_1) \wedge \texttt{neg}(F_2). 
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \vee F_2 \bigr) = \texttt{neg}(F_1) \wedge \texttt{neg}(F_2)$.</li>
</ol>

**Implementation:** The functions `nnf` and `neg` are *mutually recursive*: each of them calls the other one.  In F#, a function can only call functions that have been defined before.  Mutually recursive functions are therefore defined together using `let rec ... and ...`.  If `f` contains one of the operators `→`, `↔`, or `⊕`, an exception is raised.

In [6]:
let rec nnf (f: Formula) : Formula =
    match f with
    | Var _ | Verum | Falsum -> f
    | Not g      -> neg g
    | And (g, h) -> And (nnf g, nnf h)
    | Or  (g, h) -> Or  (nnf g, nnf h)
    | Imp _ | Iff _ | Xor _ -> failwith "nnf: →, ↔, and ⊕ have to be eliminated first"

and neg (f: Formula) : Formula =
    match f with
    | Var _      -> Not f
    | Verum      -> Falsum
    | Falsum     -> Verum
    | Not g      -> nnf g
    | And (g, h) -> Or  (neg g, neg h)
    | Or  (g, h) -> And (neg g, neg h)
    | Imp _ | Iff _ | Xor _ -> failwith "neg: →, ↔, and ⊕ have to be eliminated first"

The function `cnf` takes one argument:
- `f` is a formula that is in *negation normal form*, i.e. the negation operator is only applied to propositional variables, and `f` contains only the operators `¬`, `∧`, `∨`, `⊤`, and `⊥`.

It returns the *conjunctive normal form* of $f$ in *set notation*.  In order to achieve
this it uses the distributive law
$$ (f \wedge g) \vee (h \wedge k) \Leftrightarrow (f \vee h) \wedge (f \vee k) \wedge (g \vee h) \wedge (g \vee k). $$

**Implementation:** 
- A variable $p$ is represented by the set $\bigl\{\{p\}\bigr\}$ and a negated variable $\neg p$ by the set $\bigl\{\{\neg p\}\bigr\}$.
- $\top$ is represented by the empty set of clauses, $\bot$ by the set containing the empty clause.
- The conjunction of two formulas in CNF is the union of their clause sets.  The operator `+` computes the union of two sets.
- The disjunction of two formulas in CNF consists of all clauses $k_1 \cup k_2$ where $k_1$ is a clause of the first formula and $k_2$ is a clause of the second formula.

The type `Formula` can not express that a formula is in negation normal form.  Therefore, if `f` is not in negation normal form, we raise an exception.

In [7]:
let rec cnf (f: Formula) : CNF =
    match f with
    | Var p       -> set [ set [ Pos p ] ]
    | Verum       -> Set.empty
    | Falsum      -> set [ Set.empty ]
    | Not (Var p) -> set [ set [ Neg p ] ]
    | And (g, h)  -> cnf g + cnf h
    | Or  (g, h)  -> set [ for k1 in cnf g do for k2 in cnf h -> k1 + k2 ]
    | _           -> failwith "cnf: formula is not in negation normal form"

The function `complement` takes one argument:
- `l` is a literal.

It returns the *complement* $\overline{\,l\,}$ of `l`, i.e. we have $\overline{\,p\,} = \neg p$ and $\overline{\neg p} = p$.

In [8]:
let complement (l: Literal) : Literal =
    match l with
    | Pos p -> Neg p
    | Neg p -> Pos p

The function `isTrivial` takes one argument:
- `C` is a clause.

It returns `true` if `C` is *trivial*, i.e. if `C` contains a literal together with its complement.  A trivial clause is a tautology.
`Set.exists` checks whether some element of a set satisfies the given predicate.

In [9]:
let isTrivial (C: Clause) : bool =
    Set.exists (fun l -> C.Contains (complement l)) C

The function `simplify` takes one argument:
- `Clauses` is a set of clauses.

It returns the set of all clauses from `Clauses` that are not trivial.

In [10]:
let simplify (Clauses: CNF) : CNF =
    Set.filter (fun C -> not (isTrivial C)) Clauses

The function `normalize` takes one argument:
- `f` is a propositional formula.

It transforms `f` into *conjunctive normal form*.  Furthermore, trivial clauses are removed.  

In [11]:
let normalize (f: Formula) : CNF =
    let n1 = eliminateBiconditional f
    let n2 = eliminateConditional n1
    let n3 = nnf n2
    let n4 = cnf n3
    simplify n4

In order to print formulas in CNF in a readable way, we define a number of functions that convert literals, clauses, and sets of clauses into strings.

The function `literalToString` takes one argument:
- `l` is a literal.

It returns the string `"p"` if `l = Pos "p"` and the string `"¬p"` if `l = Neg "p"`.

In [12]:
let literalToString (l: Literal) : string =
    match l with
    | Pos p -> p
    | Neg p -> "¬" + p

The function `clauseToString` takes one argument:
- `C` is a clause.

It returns a string that represents `C` as a set of literals, e.g. `"{¬p, q}"`.  `String.concat sep xs` joins the strings in `xs` and puts the separator `sep` between them.

In [13]:
let clauseToString (C: Clause) : string =
    "{" + String.concat ", " [ for l in C -> literalToString l ] + "}"

The function `cnfToString` takes one argument:
- `Clauses` is a set of clauses.

It returns a string that represents `Clauses` as a set of sets, e.g. `"{{¬p, q}, {r}}"`.

In [14]:
let cnfToString (Clauses: CNF) : string =
    "{" + String.concat ", " [ for C in Clauses -> clauseToString C ] + "}"

The function `test` takes one argument:
- `s` is a string representing a propositional formula.

It computes the conjunctive normal form of the formula and prints it.

In [15]:
let test (s: string) : unit =
    let f = parse s
    printfn "The CNF of %s is:" s
    printfn "%s" (cnfToString (normalize f))

In [16]:
test "(¬p → q) → (p → q) → q"

The CNF of 

(¬p → q) → (p → q) → q

 is:

{}

In [17]:
test "(a → b) ↔ (¬a ∧ ¬b)"

The CNF of 

(a → b) ↔ (¬a ∧ ¬b)

 is:

{{a, ¬b}, {¬a, ¬b}, {¬b}}

In [18]:
test "(p ∧ q → r) ∨ ¬r → ¬p"

The CNF of 

(p ∧ q → r) ∨ ¬r → ¬p

 is:

{{q, ¬p}, {r, ¬p}, {¬p, ¬r}}

In [19]:
test "⊤"

The CNF of 

⊤

 is:

{}

In [20]:
test "⊥"

The CNF of 

⊥

 is:

{{}}

In [21]:
test "(p ∧ q → r) ∨ ¬r → ¬p ↔ ¬p"

The CNF of 

(p ∧ q → r) ∨ ¬r → ¬p ↔ ¬p

 is:

{}

Of course, we can also look at the clauses as values of type `CNF`:

In [22]:
normalize (parse "(a → b) ↔ (¬a ∧ ¬b)")

set [set [Pos "a"; Neg "b"]; set [Neg "a"; Neg "b"]; set [Neg "b"]]